# PlacementIQ — Notebook 04: Model Training & Evaluation

**Objective:**
1. Train 4 candidate classification algorithms: Logistic Regression, Random Forest, Extra Trees, and XGBoost.
2. Perform 5-Fold Stratified Cross-Validation on the 80,000 training partition.
3. Evaluate all candidate models on the 20,000 holdout test set across all standard classification metrics.
4. Compare models using Accuracy, Precision, Recall, F1, ROC-AUC, PR-AUC, and Brier Score.
5. Select the winning model based on documented criteria and optimize its classification threshold.
6. Serialize the fitted model pipeline for production backend inference.

## 1. Setup & Imports

In [ ]:
import sys
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json

# Ensure src directory is in path
SRC_DIR = Path('../src').resolve()
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from train import get_candidate_models, cross_validate_model, save_model
from evaluate import (
    evaluate_model, build_comparison_table, plot_confusion_matrix,
    plot_roc_curve, plot_pr_curve, plot_calibration_curve,
    find_optimal_threshold
)

sns.set_theme(style="whitegrid")
print("Environment initialized successfully.")

## 2. Load Processed Train/Test Splits

In [ ]:
X_train = pd.read_csv('../data/processed/X_train.csv')
y_train = pd.read_csv('../data/processed/y_train.csv')['placement_status']
X_test = pd.read_csv('../data/processed/X_test.csv')
y_test = pd.read_csv('../data/processed/y_test.csv')['placement_status']

print(f"Training set : {X_train.shape} features, {len(y_train)} targets (Placed: {y_train.mean():.1%})")
print(f"Test set     : {X_test.shape} features, {len(y_test)} targets (Placed: {y_test.mean():.1%})")

## 3. Stratified 5-Fold Cross-Validation Across 4 Candidate Models

In [ ]:
candidate_models = get_candidate_models()
cv_summary = {}

for name, pipeline in candidate_models.items():
    print(f"\nEvaluating {name.upper()} via 5-Fold Stratified CV...")
    res = cross_validate_model(pipeline, X_train, y_train, n_folds=5)
    cv_summary[name] = res
    print(f"  Accuracy : {res['accuracy_mean']:.4f} ± {res['accuracy_std']:.4f}")
    print(f"  ROC-AUC  : {res['roc_auc_mean']:.4f} ± {res['roc_auc_std']:.4f}")
    print(f"  F1-Score : {res['f1_mean']:.4f} ± {res['f1_std']:.4f}")

## 4. Full Training on Train Set & Holdout Test Set Evaluation

In [ ]:
test_eval_results = {}

for name, pipeline in candidate_models.items():
    print(f"Fitting {name} on full 80,000 samples...")
    pipeline.fit(X_train, y_train)
    print(f"Evaluating on 20,000 holdout test samples...")
    test_eval_results[name] = evaluate_model(pipeline, X_test, y_test, model_name=name)

## 5. Model Comparison Table

In [ ]:
comparison_table = build_comparison_table(test_eval_results)
display(comparison_table)

## 6. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for idx, (name, res) in enumerate(test_eval_results.items()):
    cm = res['confusion_matrix']
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
                xticklabels=['Not Placed (0)', 'Placed (1)'],
                yticklabels=['Not Placed (0)', 'Placed (1)'])
    axes[idx].set_title(f"Confusion Matrix: {name}", fontweight='bold')
    axes[idx].set_xlabel('Predicted')
    axes[idx].set_ylabel('True')

plt.tight_layout()
plt.show()

## 7. Comparative ROC and Precision-Recall Curves

In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve, auc

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for name, pipeline in candidate_models.items():
    y_proba = pipeline.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    roc_score = auc(fpr, tpr)
    axes[0].plot(fpr, tpr, label=f"{name} (AUC = {roc_score:.3f})")
    
    prec, rec, _ = precision_recall_curve(y_test, y_proba)
    pr_score = auc(rec, prec)
    axes[1].plot(rec, prec, label=f"{name} (PR-AUC = {pr_score:.3f})")

axes[0].plot([0, 1], [0, 1], 'k--', label='Random')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curves', fontweight='bold')
axes[0].legend()

axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curves', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()

## 8. Optimal Threshold Tuning for Winning Model

In [ ]:
winning_name = comparison_table.iloc[0]['Model']
winning_pipeline = candidate_models[winning_name]
y_test_proba = winning_pipeline.predict_proba(X_test)[:, 1]

thresh, best_f1 = find_optimal_threshold(y_test, y_test_proba, metric='f1')
print(f"Selected Model: {winning_name}")
print(f"Optimal Decision Threshold: {thresh:.2f} (F1 Score: {best_f1:.4f})")

## 9. Serialization of Final Pipeline & Artifacts

In [ ]:
saved_path = save_model(
    winning_pipeline,
    model_name=winning_name,
    metadata={
        'optimal_threshold': thresh,
        'holdout_metrics': test_eval_results[winning_name]['metrics']
    },
    threshold=thresh
)
print(f"Final model serialized successfully to {saved_path}")